In [1]:
import dimod
from dimod import ConstrainedQuadraticModel, Binary
from dimod import quicksum

# Depot, Customers 1 and 2, Vehicles a and b
nodes = [0, 1, 2]          # 0 = depot
customers = [1, 2]
vehicles = ['a', 'b']



import itertools
import random
def GenerateDummyT(Nlocations, min_cost, max_cost, symmetric):
    t = {}
    for i, j in itertools.permutations(range(Nlocations), 2):
        if symmetric and (j, i) in t:
            t[(i, j)] = t[(j, i)]
        else:
            t[(i, j)] = random.randint(min_cost, max_cost)

    return t
t = GenerateDummyT(7, 1, 5, True)




# Create binary decision variables                   
# ---------------------------------
x = {}
for i in nodes:
    for j in nodes:
        if i != j: 
            for k in vehicles:
                name = f"x_{i}_{j}_{k}"
                x[i,j,k] = Binary(name)


# Building the CQM
# -----------------
cqm = ConstrainedQuadraticModel()

# 1) Objective: minimize total travel time
objective = quicksum(t[i,j] * x[i,j,k]
                     for i,j,k in x.keys())
cqm.set_objective(objective)

# Constraints
# -------------

# Each customer visited once
for v in customers:
    cqm.add_constraint(
        quicksum(x[i,v,k] for i in nodes if i != v for k in vehicles) == 1,
        label=f"Visit_{v}"
    )

# Start and end at Depot for each vehicle
y = {}
for k in vehicles:
    y[k] = Binary(f"y_{k}")

for k in vehicles:
    # start from depot once
    cqm.add_constraint(
        sum(x[0,j,k] for j in customers)  - y[k] == 0,
        label=f"Start_{k}"
    )
    # return to depot once
    cqm.add_constraint(
        sum(x[i,0,k] for i in customers) - y[k] == 0,
        label=f"End_{k}"
    )

# Flow conservation
for v in customers:
    for k in vehicles:
        cqm.add_constraint(
            quicksum(x[i,v,k] for i in nodes if i != v) -
            quicksum(x[v,j,k] for j in nodes if j != v) == 0,
            label=f"flow_{v}_{k}"
        )

# MTZ Creating variables
u = {}
for v in customers:
    for k in vehicles:
        u[v,k] = dimod.Integer(
            f"u_{v}_{k}",
            lower_bound=0,
            upper_bound=len(customers) 
        )

# MTZ Constraint
M = len(customers)
for k in vehicles:
    for i in customers:
        for j in customers:
            if i != j:
                cqm.add_constraint(
                    u[i,k] - u[j,k] + M * x[i,j,k] <= M - 1,
                    label=f"mtz_{i}_{j}_{k}"
                )

for v in customers:
    for k in vehicles:
        cqm.add_constraint(
            u[v,k]
            - (len(customers) - 1) * quicksum(x[i,v,k] for i in nodes if i != v)
            <= 0,
            label=f"u_bind_{v}_{k}"
        )


# Converting to BQM 
bqm, invert = dimod.cqm_to_bqm(cqm)
sampler = dimod.SimulatedAnnealingSampler()
result = sampler.sample(bqm)